## SIMAP Archive → Supabase (Phase 2)

Dieses Notebook startet `simap_archive_to_supabase.py --phase 2` als Subprozess und zeigt Live-Logs.


## SIMAP Archive → Supabase (Phase 2)

Dieses Notebook startet `simap_archive_to_supabase.py --phase 2` als Subprozess und zeigt Live-Logs.


In [ ]:
import os
import signal
import subprocess
import sys
import time
from pathlib import Path

from dotenv import load_dotenv

REPO_ROOT = Path.cwd()
SCRIPT = REPO_ROOT / "simap_archive_to_supabase.py"
LOG_FILE = REPO_ROOT / ".phase2.log"
PID_FILE = REPO_ROOT / ".phase2.pid"

load_dotenv(REPO_ROOT / ".env")

assert SCRIPT.exists(), f"Script nicht gefunden: {SCRIPT}"
assert os.environ.get("SUPABASE_SERVICE_ROLE_KEY"), "SUPABASE_SERVICE_ROLE_KEY fehlt (in .env oder env vars)"
print("OK: .env geladen, Script gefunden")


In [ ]:
# Phase 2 starten (läuft im Hintergrund)
if PID_FILE.exists():
    print("Hinweis: PID-Datei existiert bereits:", PID_FILE.read_text().strip())

with open(LOG_FILE, "wb") as f:
    proc = subprocess.Popen(
        [sys.executable, str(SCRIPT), "--phase", "2"],
        stdout=f,
        stderr=subprocess.STDOUT,
        cwd=str(REPO_ROOT),
        env=os.environ.copy(),
    )

PID_FILE.write_text(str(proc.pid))
print("Gestartet. PID:", proc.pid)
print("Log:", LOG_FILE)


In [ ]:
# Live-Logs anzeigen (brich die Zelle ab, um das Tail zu stoppen)
last_size = 0
while True:
    if LOG_FILE.exists():
        data = LOG_FILE.read_bytes()
        if len(data) != last_size:
            chunk = data[last_size:]
            try:
                text = chunk.decode("utf-8", errors="replace")
            except Exception:
                text = str(chunk)
            print(text, end="")
            last_size = len(data)
    time.sleep(1)


In [ ]:
# Skript stoppen
if not PID_FILE.exists():
    raise RuntimeError("Keine PID-Datei gefunden – läuft das Skript?")

pid = int(PID_FILE.read_text().strip())
print("Stoppe PID:", pid)
try:
    os.kill(pid, signal.SIGTERM)
except ProcessLookupError:
    print("Prozess existiert nicht mehr")
finally:
    PID_FILE.unlink(missing_ok=True)
print("Stopp-Signal gesendet")
